# Generate prepared study speech with Qwen3-TTS Ryan

This notebook generates the complete prepared speech set with the **Ryan** voice from Qwen3-TTS. It reads the same fixed messages and `questions/questions.md` used by the Streamlit app and saves 18 compatible WAV files under `assets/speech/Quen3-TTS-Ryan/`.

The model runs locally through MLX on an Apple Silicon Mac. The first run downloads the model (approximately 4.5 GB); later runs reuse the cached copy. The app can use this Ryan voice set after its active speech folder is changed.

## Step 1: Install the notebook-only dependencies

These packages are installed into the active notebook kernel. They are intentionally separate from the Streamlit application's `requirements.txt`.

In [1]:
%pip install -q mlx-audio==0.5.0 soundfile==0.14.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Locate the repository and configure generation

A warm style instruction is applied only to the welcome and transition messages. The speaker test and question voiceovers use Ryan's neutral default delivery.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys
import wave

import mlx.core as mx
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from huggingface_hub import constants as huggingface_constants
from mlx_audio.tts.utils import load_model

current_directory = Path.cwd().resolve()
repo_root = current_directory.parent

if platform.system() != "Darwin" or platform.machine() != "arm64":
    raise RuntimeError("This notebook requires an Apple Silicon Mac.")

sys.path.insert(0, str(repo_root))

from services.question_loader import load_questions
from services.speech import expected_speech_assets

model_id = "mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16"
speaker = "Ryan"
language = "English"
warm_style_instruction = (
    "Speak warmly and appreciatively, with a gentle cheerful tone and a "
    "slightly upbeat ending."
)
style_instructions = {
    "welcome.wav": warm_style_instruction,
    "next_question.wav": warm_style_instruction,
    "final_question.wav": (
        "Use a clear, confident, full speaking voice at a normal conversational "
        "volume. Sound warm and encouraging; do not whisper."
    ),
    "completion.wav": warm_style_instruction,
}
random_seed = 42
output_directory = repo_root / "assets" / "speech" / "Qwen3-TTS-Ryan"

print(f"Repository: {repo_root}")
print(f"Output: {output_directory}")

Repository: /Users/tandon.utsav2/Library/CloudStorage/OneDrive-SharedLibraries-ImperialCollegeLondon/Salomons, Nicole - PAIR Lab/PAIR Lab- Code Repository/Utsav MSC Codebase/Reachy_app_laptop
Output: /Users/tandon.utsav2/Library/CloudStorage/OneDrive-SharedLibraries-ImperialCollegeLondon/Salomons, Nicole - PAIR Lab/PAIR Lab- Code Repository/Utsav MSC Codebase/Reachy_app_laptop/assets/speech/Quen3-TTS-Ryan


/Users/tandon.utsav2/Library/CloudStorage/OneDrive-SharedLibraries-ImperialCollegeLondon/Salomons, Nicole - PAIR Lab/PAIR Lab- Code Repository/Utsav MSC Codebase/Reachy_app_laptop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3: Load all study messages

The existing application helpers ensure this voice set contains the welcome, acknowledgements, practice question, and all 12 scored questions.

In [3]:
questions = load_questions(repo_root / "questions" / "questions.md")
speech_assets = expected_speech_assets(questions)

print(f"Loaded {len(questions)} questions.")
print(f"Preparing {len(speech_assets)} speech files:")
for filename in speech_assets:
    print(f"  {filename}")

Loaded 13 questions.
Preparing 18 speech files:
  speaker_test.wav
  welcome.wav
  next_question.wav
  final_question.wav
  completion.wav
  question_00.wav
  question_01.wav
  question_02.wav
  question_03.wav
  question_04.wav
  question_05.wav
  question_06.wav
  question_07.wav
  question_08.wav
  question_09.wav
  question_10.wav
  question_11.wav
  question_12.wav


## Step 4: Download and load Qwen3-TTS

The initial execution downloads the model. Loading it once allows every study message to be generated in the same run.

In [4]:
# This public model does not need the cached Hugging Face login token.
# Disabling implicit authentication also avoids failures from an expired token.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
huggingface_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True

print(f"Loading {model_id}...")
model = load_model(model_id)
print("Model loaded.")

Loading mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16...


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 211477.51it/s]
[transformers] You are using a model of type `qwen3_tts` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


  Initialized encoder codebooks
Loaded speech tokenizer from /Users/tandon.utsav2/.cache/huggingface/hub/models--mlx-community--Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16/snapshots/52f4770fd9726457eae3d3b6aa92047a25a10776/speech_tokenizer
Model loaded.


## Step 5: Generate and save all Ryan WAV files

Existing files with the same names in the Ryan folder are replaced. The Aiden and default macOS voice folders are not modified.

In [5]:
output_directory.mkdir(parents=True, exist_ok=True)
mx.random.seed(random_seed)

for number, (filename, text) in enumerate(speech_assets.items(), start=1):
    print(f"[{number}/{len(speech_assets)}] Generating {filename}")
    instruction = style_instructions.get(filename)
    results = list(
        model.generate_custom_voice(
            text=text,
            speaker=speaker,
            language=language,
            instruct=instruction,
        )
    )
    if not results:
        raise RuntimeError(f"The model returned no audio for {filename}.")

    sample_rates = {result.sample_rate for result in results}
    if sample_rates != {24_000}:
        raise RuntimeError(
            f"Unexpected sample rate for {filename}: {sorted(sample_rates)}"
        )

    audio_parts = [np.asarray(result.audio).squeeze() for result in results]
    if any(audio.ndim != 1 or audio.size == 0 for audio in audio_parts):
        raise RuntimeError(f"The model returned invalid audio for {filename}.")
    audio = np.concatenate(audio_parts)

    output_path = output_directory / filename
    temporary_path = output_directory / f".{Path(filename).stem}.tmp.wav"
    sf.write(temporary_path, audio, 24_000, subtype="PCM_16")
    temporary_path.replace(output_path)

(output_directory / "manifest.json").write_text(
    json.dumps(speech_assets, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
(output_directory / "generation.json").write_text(
    json.dumps(
        {
            "model": model_id,
            "speaker": speaker,
            "language": language,
            "default_style_instruction": None,
            "file_style_instructions": style_instructions,
            "random_seed": random_seed,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Generated {len(speech_assets)} files in {output_directory}")

[1/18] Generating speaker_test.wav
[2/18] Generating welcome.wav
[3/18] Generating next_question.wav
[4/18] Generating final_question.wav
[5/18] Generating completion.wav
[6/18] Generating question_00.wav
[7/18] Generating question_01.wav
[8/18] Generating question_02.wav
[9/18] Generating question_03.wav
[10/18] Generating question_04.wav
[11/18] Generating question_05.wav
[12/18] Generating question_06.wav
[13/18] Generating question_07.wav
[14/18] Generating question_08.wav
[15/18] Generating question_09.wav
[16/18] Generating question_10.wav
[17/18] Generating question_11.wav
[18/18] Generating question_12.wav
Generated 18 files in /Users/tandon.utsav2/Library/CloudStorage/OneDrive-SharedLibraries-ImperialCollegeLondon/Salomons, Nicole - PAIR Lab/PAIR Lab- Code Repository/Utsav MSC Codebase/Reachy_app_laptop/assets/speech/Quen3-TTS-Ryan


## Step 6: Validate the generated voice set

Every file must be a non-empty mono, 16-bit, 24 kHz WAV before it can be used by the study application.

In [6]:
validation_rows = []

for filename in speech_assets:
    audio_path = output_directory / filename
    with wave.open(str(audio_path), "rb") as audio_file:
        channels = audio_file.getnchannels()
        sample_width = audio_file.getsampwidth()
        sample_rate = audio_file.getframerate()
        frame_count = audio_file.getnframes()

    valid = (
        channels == 1
        and sample_width == 2
        and sample_rate == 24_000
        and frame_count > 0
    )
    if not valid:
        raise RuntimeError(f"Invalid generated WAV: {audio_path}")

    validation_rows.append(
        {
            "filename": filename,
            "duration_seconds": round(frame_count / sample_rate, 2),
        }
    )

print(f"Validated {len(validation_rows)} Ryan speech files.")
validation_rows

Validated 18 Ryan speech files.


[{'filename': 'speaker_test.wav', 'duration_seconds': 2.56},
 {'filename': 'welcome.wav', 'duration_seconds': 20.56},
 {'filename': 'next_question.wav', 'duration_seconds': 3.6},
 {'filename': 'final_question.wav', 'duration_seconds': 3.04},
 {'filename': 'completion.wav', 'duration_seconds': 4.16},
 {'filename': 'question_00.wav', 'duration_seconds': 10.8},
 {'filename': 'question_01.wav', 'duration_seconds': 32.24},
 {'filename': 'question_02.wav', 'duration_seconds': 24.48},
 {'filename': 'question_03.wav', 'duration_seconds': 29.36},
 {'filename': 'question_04.wav', 'duration_seconds': 23.12},
 {'filename': 'question_05.wav', 'duration_seconds': 17.6},
 {'filename': 'question_06.wav', 'duration_seconds': 21.2},
 {'filename': 'question_07.wav', 'duration_seconds': 19.84},
 {'filename': 'question_08.wav', 'duration_seconds': 27.76},
 {'filename': 'question_09.wav', 'duration_seconds': 29.92},
 {'filename': 'question_10.wav', 'duration_seconds': 23.68},
 {'filename': 'question_11.wav'